In [ ]:
import cv2  
import math 
import csv
import os
import numpy as np
import torch
import matplotlib.pyplot as plt

from tqdm import tqdm
from collections import defaultdict
from ultralytics import YOLO
from ultralytics import RTDETR
import re
from PIL import Image
from transformers import AutoProcessor
from transformers import AutoModelForImageTextToText

In [ ]:

model = YOLO("yolo11l.pt")
model.train(
    data=r"C:\Users\alejandro\Desktop\ULPGC\4_Carrera\VC\Practicas\P4\P4\TGC_RBNW\data.yml",
    imgsz=640,
    batch=32,
    device=0,
    epochs=100,
    workers = 0,
    verbose = True,
    plots = True
)


In [ ]:
_original_torch_load = torch.load

def _patched_torch_load(*args, **kwargs):
    if "weights_only" in kwargs:
        kwargs.pop("weights_only")
    return _original_torch_load(*args, **kwargs)

torch.load = _patched_torch_load

base_dir = r"C:\Users\alejandro\Desktop\ULPGC\4_Carrera\VC\Practicas\P4\P4"

# Vídeo de entrada
video_path = os.path.join(base_dir, "C0142.mp4")

# Pesos del modelo YOLO entrenado
yolo_model_path = os.path.join(
    base_dir, "runs", "detect", "train-28", "weights", "best.pt"
)

# Carpeta y salidas diferenciadas
output_folder = os.path.join(base_dir, "out")
os.makedirs(output_folder, exist_ok=True)
output_video_path = os.path.join(output_folder, "video_con_detecciones.mp4")
csv_file_path = os.path.join(output_folder, "resultados_deteccion.csv")

custom_model = YOLO(yolo_model_path)
coco_model = YOLO("yolo11l.pt")


# Clases COCO a detectar: personas y vehículos
person_class_index = 0
vehicle_class_indices = [2, 3, 5, 7]  # car, motorcycle, bus, truck
coco_class_indices = [person_class_index] + vehicle_class_indices

# Diccionario para contar los objetos únicos detectados
class_counts = {}

object_id_counter = 0
prev_centroids = {}

def get_centroid(box):
    x1, y1, x2, y2 = map(int, box)
    return int((x1 + x2) / 2), int((y1 + y2) / 2)

def assign_id(cx, cy, centroids_dict, threshold=50):
    global object_id_counter
    for oid, (pcx, pcy) in centroids_dict.items():
        if np.sqrt((cx - pcx) ** 2 + (cy - pcy) ** 2) < threshold:
            centroids_dict[oid] = (cx, cy)
            return oid
    object_id_counter += 1
    centroids_dict[object_id_counter] = (cx, cy)
    return object_id_counter

def draw_coco_boxes(results, frame, color=(0, 255, 0), filter_classes=None):
    for r in results:
        if r.boxes is None:
            continue
        for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
            cls = int(cls)
            if filter_classes and cls not in filter_classes:
                continue
            x1, y1, x2, y2 = map(int, box)
            label = f"{r.names[cls]} {float(conf):.2f}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(
                frame, label, (x1, max(20, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2,
            )
    return frame


cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo cargar el vídeo en: {video_path}")

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_writer = cv2.VideoWriter(
    output_video_path, fourcc, fps, (frame_width, frame_height)
)

# Cabeceras csv
csv_headers = [
    "fotograma", "tipo_objeto", "confianza", "identificador_tracking", 
    "x1", "y1", "x2", "y2", 
    "matricula_en_su_caso", "confianza_matricula", 
    "mx1", "my1", "mx2", "my2", "texto_matricula"
]

f = open(csv_file_path, mode="w", newline="", encoding="utf-8")
writer = csv.writer(f)
writer.writerow(csv_headers)

frame_num = 0
print(f"Iniciando procesamiento con YOLO")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_num += 1
    if frame_num % 50 == 0:
        print(f"Frame {frame_num}")

    coco_results = coco_model.predict(
        frame, imgsz=640, conf=0.25, verbose=False, device=0
    )

    for r in coco_results:
        if r.boxes is None:
            continue
        for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
            cls = int(cls)
            if cls not in coco_class_indices:
                continue

            x1, y1, x2, y2 = map(int, box)
            cx, cy = get_centroid((x1, y1, x2, y2))
            oid = assign_id(cx, cy, prev_centroids)
            tipo_obj = r.names[cls]

            if tipo_obj not in class_counts:
                class_counts[tipo_obj] = set()
            class_counts[tipo_obj].add(oid)

            writer.writerow([
                frame_num, tipo_obj, f"{float(conf):.3f}", oid,
                x1, y1, x2, y2,
                "", "", "", "", "", "", ""
            ])

    custom_results = custom_model.predict(
        frame, imgsz=640, conf=0.25, verbose=False, device=0
    )

    for r in custom_results:
        if r.boxes is None:
            continue
        for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
            x1, y1, x2, y2 = map(int, box)

            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(frame_width, x2), min(frame_height, y2)

            cx, cy = get_centroid((x1, y1, x2, y2))
            oid = assign_id(cx, cy, prev_centroids)
            tipo_obj = r.names[int(cls)] 

            if tipo_obj not in class_counts:
                class_counts[tipo_obj] = set()
            class_counts[tipo_obj].add(oid)

            writer.writerow([
                frame_num, "vehiculo_con_matricula", "", oid,  
                "", "", "", "", 
                tipo_obj, f"{float(conf):.3f}", 
                x1, y1, x2, y2, 
                "" 
            ])

            label = f"{tipo_obj} {float(conf):.2f}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
            cv2.putText(
                frame, label, (x1, max(20, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2,
            )

    frame = draw_coco_boxes(
        coco_results, frame, color=(0, 255, 0), filter_classes=coco_class_indices
    )

    y_offset = 35
    cv2.putText(frame, "Cuenta Total:", (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2, cv2.LINE_AA)
    for obj_class, ids in class_counts.items():
        y_offset += 30
        text = f"{obj_class}: {len(ids)}"
        cv2.putText(frame, text, (11, y_offset + 1), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2, cv2.LINE_AA)
        cv2.putText(frame, text, (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2, cv2.LINE_AA)

    out_writer.write(frame)


cap.release()
out_writer.release()
f.close()

In [ ]:
rtdetr_model = RTDETR("rtdetr-l.pt")
rtdetr_train_results = rtdetr_model.train(
    data=r"C:\Users\alejandro\Desktop\ULPGC\4_Carrera\VC\Practicas\P4\P4\TGC_RBNW\data.yml",
        imgsz=640,
        batch=16,
        device=0,
        epochs=100,
        workers=0,
        verbose=True,
        plots=True,
        project="runs_matriculas",
        name="rtdetr_matriculas"
    )

In [ ]:
_original_torch_load = torch.load

def _patched_torch_load(*args, **kwargs):
    if "weights_only" in kwargs:
        kwargs.pop("weights_only")
    return _original_torch_load(*args, **kwargs)

torch.load = _patched_torch_load
base_dir = r"C:\Users\alejandro\Desktop\ULPGC\4_Carrera\VC\Practicas\P4\P4"

video_path = os.path.join(base_dir, "C0142.mp4")

rtdetr_model_path = os.path.join(
    base_dir, "runs", "detect", "runs_matriculas", "rtdetr_matriculas-2", "weights", "best.pt"
)

output_folder = os.path.join(base_dir, "out")
os.makedirs(output_folder, exist_ok=True)
output_video_path = os.path.join(output_folder, "video_con_detecciones_rtdetr.mp4")
csv_file_path = os.path.join(output_folder, "resultados_deteccion_rtdetr.csv")

custom_model = RTDETR(rtdetr_model_path)
coco_model = YOLO("yolo11l.pt")

person_class_index = 0
vehicle_class_indices = [2, 3, 5, 7]
coco_class_indices = [person_class_index] + vehicle_class_indices

class_counts = {}

object_id_counter = 0
prev_centroids = {}

def get_centroid(box):
    x1, y1, x2, y2 = map(int, box)
    return int((x1 + x2) / 2), int((y1 + y2) / 2)

def assign_id(cx, cy, centroids_dict, threshold=50):
    global object_id_counter
    for oid, (pcx, pcy) in centroids_dict.items():
        if np.sqrt((cx - pcx) ** 2 + (cy - pcy) ** 2) < threshold:
            centroids_dict[oid] = (cx, cy)
            return oid
    object_id_counter += 1
    centroids_dict[object_id_counter] = (cx, cy)
    return object_id_counter

def draw_coco_boxes(results, frame, color=(0, 255, 0), filter_classes=None):
    for r in results:
        if r.boxes is None:
            continue
        for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
            cls = int(cls)
            if filter_classes and cls not in filter_classes:
                continue
            x1, y1, x2, y2 = map(int, box)
            label = f"{r.names[cls]} {float(conf):.2f}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(
                frame, label, (x1, max(20, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2,
            )
    return frame


cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo cargar el vídeo en: {video_path}")

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_writer = cv2.VideoWriter(
    output_video_path, fourcc, fps, (frame_width, frame_height)
)

csv_headers = [
    "fotograma", "tipo_objeto", "confianza", "identificador_tracking", 
    "x1", "y1", "x2", "y2", 
    "matricula_en_su_caso", "confianza_matricula", 
    "mx1", "my1", "mx2", "my2", "texto_matricula"
]

f = open(csv_file_path, mode="w", newline="", encoding="utf-8")
writer = csv.writer(f)
writer.writerow(csv_headers)


frame_num = 0
print(f"Iniciando procesamiento con RT-DETR")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_num += 1
    if frame_num % 50 == 0:
        print(f"Frame {frame_num}")

    coco_results = coco_model.predict(
        frame, imgsz=640, conf=0.25, verbose=False, device=0
    )

    for r in coco_results:
        if r.boxes is None:
            continue
        for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
            cls = int(cls)
            if cls not in coco_class_indices:
                continue

            x1, y1, x2, y2 = map(int, box)
            cx, cy = get_centroid((x1, y1, x2, y2))
            oid = assign_id(cx, cy, prev_centroids)
            tipo_obj = r.names[cls]

            if tipo_obj not in class_counts:
                class_counts[tipo_obj] = set()
            class_counts[tipo_obj].add(oid)

            writer.writerow([
                frame_num, tipo_obj, f"{float(conf):.3f}", oid,
                x1, y1, x2, y2,
                "", "", "", "", "", "", ""
            ])

    custom_results = custom_model.predict(
        frame, imgsz=640, conf=0.25, verbose=False, device=0
    )

    for r in custom_results:
        if r.boxes is None:
            continue
        for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
            x1, y1, x2, y2 = map(int, box)

            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(frame_width, x2), min(frame_height, y2)

            cx, cy = get_centroid((x1, y1, x2, y2))
            oid = assign_id(cx, cy, prev_centroids)
            tipo_obj = r.names[int(cls)]

            if tipo_obj not in class_counts:
                class_counts[tipo_obj] = set()
            class_counts[tipo_obj].add(oid)

            writer.writerow([
                frame_num, "vehiculo_con_matricula", "", oid,  
                "", "", "", "", 
                tipo_obj, f"{float(conf):.3f}", 
                x1, y1, x2, y2, 
                "" 
            ])

            label = f"{tipo_obj} {float(conf):.2f}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 255), 2)
            cv2.putText(
                frame, label, (x1, max(20, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 255), 2,
            )

    frame = draw_coco_boxes(
        coco_results, frame, color=(0, 255, 0), filter_classes=coco_class_indices
    )

    y_offset = 35
    cv2.putText(frame, "Cuenta Total:", (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2, cv2.LINE_AA)
    for obj_class, ids in class_counts.items():
        y_offset += 30
        text = f"{obj_class}: {len(ids)}"
        cv2.putText(frame, text, (11, y_offset + 1), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2, cv2.LINE_AA)
        cv2.putText(frame, text, (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2, cv2.LINE_AA)

    out_writer.write(frame)

cap.release()
out_writer.release()
f.close()

In [ ]:
yolo_csv = os.path.join(BASE_DIR, "runs", "detect", "train-28", "results.csv")
rtdetr_csv = os.path.join(BASE_DIR, "runs", "detect", "runs_matriculas", "rtdetr_matriculas-2", "results.csv")

fig, axs = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("Evolución del Entrenamiento: YOLO11-Large vs RT-DETR-Large (100 Épocas)", fontsize=16, fontweight='bold')

# Pérdidas de entrenamiento
if os.path.exists(yolo_csv):
    df_yolo = pd.read_csv(yolo_csv)
    df_yolo.columns = df_yolo.columns.str.strip()
    axs[0, 0].plot(df_yolo['epoch'], df_yolo['train/box_loss'], label='YOLO Train Box Loss', color='royalblue', lw=2)
    if 'val/box_loss' in df_yolo.columns:
        axs[0, 0].plot(df_yolo['epoch'], df_yolo['val/box_loss'], label='YOLO Val Box Loss', color='royalblue', linestyle='--', lw=1.5)

if os.path.exists(rtdetr_csv):
    df_rtdetr = pd.read_csv(rtdetr_csv)
    df_rtdetr.columns = df_rtdetr.columns.str.strip()
    axs[0, 0].plot(df_rtdetr['epoch'], df_rtdetr['train/giou_loss'], label='RT-DETR Train GIoU Loss', color='crimson', lw=2)
    if 'val/giou_loss' in df_rtdetr.columns:
        axs[0, 0].plot(df_rtdetr['epoch'], df_rtdetr['val/giou_loss'], label='RT-DETR Val GIoU Loss', color='crimson', linestyle='--', lw=1.5)

axs[0, 0].set_title("Función de Pérdida de Regresión de Cajas")
axs[0, 0].set_xlabel("Época")
axs[0, 0].set_ylabel("Pérdida (Loss)")
axs[0, 0].grid(True, alpha=0.3)
axs[0, 0].legend()

# Evolución de mAP@50
if os.path.exists(yolo_csv):
    axs[0, 1].plot(df_yolo['epoch'], df_yolo['metrics/mAP50(B)'], label='YOLO mAP@50', color='royalblue', lw=2)
if os.path.exists(rtdetr_csv):
    axs[0, 1].plot(df_rtdetr['epoch'], df_rtdetr['metrics/mAP50(B)'], label='RT-DETR mAP@50', color='crimson', lw=2)
axs[0, 1].set_title("Evolución de mAP@50 (Validación)")
axs[0, 1].set_xlabel("Época")
axs[0, 1].set_ylabel("mAP@50")
axs[0, 1].grid(True, alpha=0.3)
axs[0, 1].legend()

# Evolución de mAP@50-95
if os.path.exists(yolo_csv):
    axs[1, 0].plot(df_yolo['epoch'], df_yolo['metrics/mAP50-95(B)'], label='YOLO mAP@50-95', color='royalblue', lw=2)
if os.path.exists(rtdetr_csv):
    axs[1, 0].plot(df_rtdetr['epoch'], df_rtdetr['metrics/mAP50-95(B)'], label='RT-DETR mAP@50-95', color='crimson', lw=2)
axs[1, 0].set_title("Evolución de mAP@50-95 (Validación)")
axs[1, 0].set_xlabel("Época")
axs[1, 0].set_ylabel("mAP@50-95")
axs[1, 0].grid(True, alpha=0.3)
axs[1, 0].legend()

# Precisión y Recall
if os.path.exists(yolo_csv):
    axs[1, 1].plot(df_yolo['epoch'], df_yolo['metrics/precision(B)'], label='YOLO Precisión', color='royalblue', lw=1.8)
    axs[1, 1].plot(df_yolo['epoch'], df_yolo['metrics/recall(B)'], label='YOLO Recall', color='deepskyblue', linestyle='--', lw=1.8)
if os.path.exists(rtdetr_csv):
    axs[1, 1].plot(df_rtdetr['epoch'], df_rtdetr['metrics/precision(B)'], label='RT-DETR Precisión', color='crimson', lw=1.8)
    axs[1, 1].plot(df_rtdetr['epoch'], df_rtdetr['metrics/recall(B)'], label='RT-DETR Recall', color='lightcoral', linestyle='--', lw=1.8)
axs[1, 1].set_title("Precisión y Exhaustividad (Recall)")
axs[1, 1].set_xlabel("Época")
axs[1, 1].set_ylabel("Valor [0 - 1]")
axs[1, 1].grid(True, alpha=0.3)
axs[1, 1].legend()

plt.tight_layout()
plt.show()


In [ ]:
try:
    import ultralytics.nn.tasks
    torch.serialization.add_safe_globals([ultralytics.nn.tasks.DetectionModel])
except Exception as e:
    print(f"Nota: No se pudo registrar la clase global: {e}")

base_dir = r"" # Al que lo ejecute, poner el directorio raíz donde guarde los archivos
video_path = os.path.join(base_dir, "C0142.MP4")

yolo_model_path = os.path.join(
    base_dir,
    "bestYOLO.pt",
)

output_folder = os.path.join("out")
os.makedirs(output_folder, exist_ok=True)
output_video_path = os.path.join(
    output_folder, "video_con_detecciones_ocr_yolo.mp4"
)
csv_file_path = os.path.join(output_folder, "resultsOCR_yolo.csv")

# Cargamos los modelos
custom_model = YOLO(yolo_model_path)

print("Cargando detector COCO (YOLO11-Large)...")
coco_model = YOLO("yolo11l.pt")

ocr_reader = easyocr.Reader(["es", "en"], gpu=True)
print("Modelos inicializados correctamente.")

person_class_index = 0
vehicle_class_indices = [2, 3, 5, 7]
coco_class_indices = [person_class_index] + vehicle_class_indices

OCR_MARGEN = 0.08  # Margen por lado para el recorte de la matrícula


def limitar_caja(caja, ancho, alto):
    x1, y1, x2, y2 = map(int, caja)
    return (
        max(0, min(ancho, x1)),
        max(0, min(alto, y1)),
        max(0, min(ancho, x2)),
        max(0, min(alto, y2)),
    )


# Generador de variantes: original -> CLAHE -> Binarización Otsu.
def variantes_ocr(recorte):
    yield recorte
    if recorte is None or recorte.size == 0:
        return
    gris = cv2.cvtColor(recorte, cv2.COLOR_BGR2GRAY)
    contraste = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4)).apply(gris)
    yield cv2.cvtColor(contraste, cv2.COLOR_GRAY2BGR)
    _, binaria = cv2.threshold(contraste, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    yield cv2.cvtColor(binaria, cv2.COLOR_GRAY2BGR)


# Separa una matrícula de dos filas en superior e inferior de forma heurística.
def separar_filas(recorte):
    gris = cv2.cvtColor(recorte, cv2.COLOR_BGR2GRAY)
    _, tinta = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    alto, ancho = gris.shape
    inicio, fin = int(alto * 0.35), int(alto * 0.65)
    centro = tinta[:, int(ancho * 0.1):max(int(ancho * 0.9), int(ancho * 0.1) + 1)]
    densidad = np.mean(centro > 0, axis=1)
    candidatos = np.arange(inicio, max(inicio + 1, fin))
    corte = min(candidatos, key=lambda y: (densidad[y], abs(y - alto / 2)))
    return recorte[:corte], recorte[corte:]


def recognize_plate_ocr(plate_image, reader):
    try:
        if plate_image is None or plate_image.size == 0:
            return "OCR_ERROR"

        alto_p, ancho_p = plate_image.shape[:2]

        # Eliminar franja azul izquierda si la tuviera (aprox. 14% del ancho)
        ancho_corte_izq = int(ancho_p * 0.14)
        if ancho_p - ancho_corte_izq > 20:
            plate_image = plate_image[:, ancho_corte_izq:]
            alto_p, ancho_p = plate_image.shape[:2]

        doble_fila = (ancho_p / max(1, alto_p)) < 2.2

        # Escalar si es muy pequeña (altura mínima de 96 píxeles)
        escala = max(1, 96 / max(1, alto_p))
        plate_resized = cv2.resize(plate_image, None, fx=escala, fy=escala, interpolation=cv2.INTER_CUBIC)

        intentos_imagenes = []
        for var_img in variantes_ocr(plate_resized):
            if doble_fila:
                try:
                    f_sup, f_inf = separar_filas(var_img)
                    intentos_imagenes.append([
                        cv2.copyMakeBorder(f_sup, 8, 8, 8, 8, cv2.BORDER_REPLICATE),
                        cv2.copyMakeBorder(f_inf, 8, 8, 8, 8, cv2.BORDER_REPLICATE)
                    ])
                except Exception:
                    intentos_imagenes.append([var_img])
            else:
                intentos_imagenes.append([var_img])

        texto_matricula = ""
        for grupo in intentos_imagenes:
            texto_parcial = ""
            exito_grupo = True
            for img_linea in grupo:
                resultado_ocr = reader.readtext(img_linea)
                if not resultado_ocr:
                    exito_grupo = False
                    break
                sub_texto = "".join([text for (_, text, _) in resultado_ocr])
                texto_parcial += sub_texto

            if exito_grupo and texto_parcial:
                texto_matricula = texto_parcial
                break

        # Limpiar caracteres no alfanuméricos y pasar a mayúsculas
        clean = "".join(filter(str.isalnum, texto_matricula)).upper()

        if len(clean) < 7:
            return "OCR_ERROR"

        # Buscar patrón de matrículas modernas españolas (4 Dígitos + 3 Letras)
        match = re.search(r'([0-9OQDILSZEAGBT]{4})([0-9OIZEA5G7BP]{3})', clean)

        if match:
            part_num, part_let = match.groups()
        else:
            if len(clean) >= 7:
                clean = clean[-7:]
                part_num, part_let = clean[:4], clean[4:]
            else:
                return "OCR_ERROR"

        # Mapas de corrección estricta para el formato español (Números vs Letras)
        digit_map = {'O': '0', 'Q': '0', 'D': '0', 'I': '1', 'L': '1', 'Z': '2', 'E': '3', 'A': '4', 'S': '5', 'G': '6', 'B': '8', 'T': '7'}
        letter_map = {'0': 'O', '1': 'I', '2': 'Z', '3': 'E', '4': 'A', '5': 'S', '6': 'G', '7': 'T', '8': 'B', '9': 'P'}

        chars_num = [digit_map.get(c, c) for c in part_num]
        chars_let = [letter_map.get(c, c) for c in part_let]

        resultado_final = "".join(chars_num + chars_let)
        return resultado_final if len(resultado_final) == 7 else "OCR_ERROR"

    except Exception:
        return "OCR_ERROR"


object_id_counter = 0
prev_centroids = {}


def get_centroid(box):
    x1, y1, x2, y2 = map(int, box)
    return int((x1 + x2) / 2), int((y1 + y2) / 2)


def assign_id(cx, cy, centroids_dict, threshold=50):
    global object_id_counter
    for oid, (pcx, pcy) in centroids_dict.items():
        if np.sqrt((cx - pcx) ** 2 + (cy - pcy) ** 2) < threshold:
            centroids_dict[oid] = (cx, cy)
            return oid
    object_id_counter += 1
    centroids_dict[object_id_counter] = (cx, cy)
    return object_id_counter


def draw_coco_boxes(results, frame, color=(0, 255, 0), filter_classes=None):
    for r in results:
        if r.boxes is None:
            continue
        for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
            cls = int(cls)
            if filter_classes and cls not in filter_classes:
                continue
            x1, y1, x2, y2 = map(int, box)
            label = f"{r.names[cls]} {float(conf):.2f}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(
                frame,
                label,
                (x1, max(20, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2,
            )
    return frame


cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo cargar el vídeo en: {video_path}")

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_writer = cv2.VideoWriter(
    output_video_path, fourcc, fps, (frame_width, frame_height)
)

csv_headers = [
    "fotograma",
    "tipo_objeto",
    "confianza",
    "id_tracking",
    "x1",
    "y1",
    "x2",
    "y2",
    "matricula_en_su_caso",
    "conf_matricula",
    "mx1",
    "my1",
    "mx2",
    "my2",
    "texto_OCR",
]

f = open(csv_file_path, mode="w", newline="", encoding="utf-8")
writer = csv.writer(f)
writer.writerow(csv_headers)

frame_num = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_num += 1
    if frame_num % 50 == 0:
        print(f"Procesando frame {frame_num}...")

    coco_results = coco_model.predict(
        frame, imgsz=640, conf=0.25, verbose=False, device=0
    )

    for r in coco_results:
        if r.boxes is None:
            continue
        for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
            cls = int(cls)
            if cls not in coco_class_indices:
                continue

            x1, y1, x2, y2 = map(int, box)
            cx, cy = get_centroid((x1, y1, x2, y2))
            oid = assign_id(cx, cy, prev_centroids)
            tipo_obj = r.names[cls]

            writer.writerow(
                [
                    frame_num,
                    tipo_obj,
                    f"{float(conf):.3f}",
                    oid,
                    x1,
                    y1,
                    x2,
                    y2,
                    "",
                    "",
                    "",
                    "",
                    "",
                    "",
                    "",
                ]
            )

    custom_results = custom_model.predict(
        frame, imgsz=640, conf=0.25, verbose=False, device=0
    )

    for r in custom_results:
        if r.boxes is None:
            continue
        for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
            x1, y1, x2, y2 = map(int, box)

            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(frame_width, x2), min(frame_height, y2)

            cx, cy = get_centroid((x1, y1, x2, y2))
            oid = assign_id(cx, cy, prev_centroids)

            plate_crop = frame[y1:y2, x1:x2]
            texto_ocr = recognize_plate_ocr(plate_crop, ocr_reader)

            writer.writerow(
                [
                    frame_num,
                    r.names[int(cls)],
                    f"{float(conf):.3f}",
                    oid,
                    x1,
                    y1,
                    x2,
                    y2,
                    "plate",
                    f"{float(conf):.3f}",
                    x1,
                    y1,
                    x2,
                    y2,
                    texto_ocr,
                ]
            )

            label = f"{r.names[int(cls)]} {float(conf):.2f} | OCR: {texto_ocr}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
            cv2.putText(
                frame,
                label,
                (x1, max(20, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 0, 255),
                2,
            )

    frame = draw_coco_boxes(
        coco_results, frame, color=(0, 255, 0), filter_classes=coco_class_indices
    )

    out_writer.write(frame)

cap.release()
out_writer.release()
f.close()
print("Procesamiento completado con éxito.")

In [ ]:
try:
    import ultralytics.nn.tasks
    torch.serialization.add_safe_globals([ultralytics.nn.tasks.RTDETRDetectionModel])
except Exception as e:
    print(f"Nota: No se pudo registrar la clase global: {e}")

base_dir = r"" # Al que lo ejecute, poner el directorio raíz donde guarde los archivos
video_path = os.path.join(base_dir, "C0142.MP4")

yolo_model_path = os.path.join(
    base_dir,
    "bestRT.pt",
)

output_folder = os.path.join("out")
os.makedirs(output_folder, exist_ok=True)
output_video_path = os.path.join(
    output_folder, "video_con_detecciones_ocr_rtdetr.mp4"
)
csv_file_path = os.path.join(output_folder, "resultsOCR_rtdetr.csv")

# Cargamos los modelos
custom_model = RTDETR(rtdetr_model_path)

print("Cargando detector COCO (YOLO11-Large)...")
coco_model = YOLO("yolo11l.pt")

ocr_reader = easyocr.Reader(["es", "en"], gpu=True)
print("Modelos inicializados correctamente.")

person_class_index = 0
vehicle_class_indices = [2, 3, 5, 7]
coco_class_indices = [person_class_index] + vehicle_class_indices

OCR_MARGEN = 0.08  # Margen por lado para el recorte de la matrícula


def limitar_caja(caja, ancho, alto):
    x1, y1, x2, y2 = map(int, caja)
    return (
        max(0, min(ancho, x1)),
        max(0, min(alto, y1)),
        max(0, min(ancho, x2)),
        max(0, min(alto, y2)),
    )


# Generador de variantes: original -> CLAHE -> Binarización Otsu.
def variantes_ocr(recorte):
    yield recorte
    if recorte is None or recorte.size == 0:
        return
    gris = cv2.cvtColor(recorte, cv2.COLOR_BGR2GRAY)
    contraste = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4)).apply(gris)
    yield cv2.cvtColor(contraste, cv2.COLOR_GRAY2BGR)
    _, binaria = cv2.threshold(contraste, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    yield cv2.cvtColor(binaria, cv2.COLOR_GRAY2BGR)


# Separa una matrícula de dos filas en superior e inferior de forma heurística.
def separar_filas(recorte):
    gris = cv2.cvtColor(recorte, cv2.COLOR_BGR2GRAY)
    _, tinta = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    alto, ancho = gris.shape
    inicio, fin = int(alto * 0.35), int(alto * 0.65)
    centro = tinta[:, int(ancho * 0.1):max(int(ancho * 0.9), int(ancho * 0.1) + 1)]
    densidad = np.mean(centro > 0, axis=1)
    candidatos = np.arange(inicio, max(inicio + 1, fin))
    corte = min(candidatos, key=lambda y: (densidad[y], abs(y - alto / 2)))
    return recorte[:corte], recorte[corte:]


def recognize_plate_ocr(plate_image, reader):
    try:
        if plate_image is None or plate_image.size == 0:
            return "OCR_ERROR"

        alto_p, ancho_p = plate_image.shape[:2]

        # Eliminar franja azul izquierda si la tuviera (aprox. 14% del ancho)
        ancho_corte_izq = int(ancho_p * 0.14)
        if ancho_p - ancho_corte_izq > 20:
            plate_image = plate_image[:, ancho_corte_izq:]
            alto_p, ancho_p = plate_image.shape[:2]

        doble_fila = (ancho_p / max(1, alto_p)) < 2.2

        # Escalar si es muy pequeña (altura mínima de 96 píxeles)
        escala = max(1, 96 / max(1, alto_p))
        plate_resized = cv2.resize(plate_image, None, fx=escala, fy=escala, interpolation=cv2.INTER_CUBIC)

        intentos_imagenes = []
        for var_img in variantes_ocr(plate_resized):
            if doble_fila:
                try:
                    f_sup, f_inf = separar_filas(var_img)
                    intentos_imagenes.append([
                        cv2.copyMakeBorder(f_sup, 8, 8, 8, 8, cv2.BORDER_REPLICATE),
                        cv2.copyMakeBorder(f_inf, 8, 8, 8, 8, cv2.BORDER_REPLICATE)
                    ])
                except Exception:
                    intentos_imagenes.append([var_img])
            else:
                intentos_imagenes.append([var_img])

        texto_matricula = ""
        for grupo in intentos_imagenes:
            texto_parcial = ""
            exito_grupo = True
            for img_linea in grupo:
                # EasyOCR devueve una lista de tuplas: (bbox, text, prob)
                resultado_ocr = reader.readtext(img_linea)
                if not resultado_ocr:
                    exito_grupo = False
                    break
                sub_texto = "".join([text for (_, text, _) in resultado_ocr])
                texto_parcial += sub_texto

            if exito_grupo and texto_parcial:
                texto_matricula = texto_parcial
                break

        # Limpiar caracteres no alfanuméricos y pasar a mayúsculas
        clean = "".join(filter(str.isalnum, texto_matricula)).upper()

        if len(clean) < 7:
            return "OCR_ERROR"

        # Buscar patrón de matrículas modernas españolas (4 Dígitos + 3 Letras)
        match = re.search(r'([0-9OQDILSZEAGBT]{4})([0-9OIZEA5G7BP]{3})', clean)

        if match:
            part_num, part_let = match.groups()
        else:
            if len(clean) >= 7:
                clean = clean[-7:]
                part_num, part_let = clean[:4], clean[4:]
            else:
                return "OCR_ERROR"

        # Mapas de corrección estricta para el formato español (Números vs Letras)
        digit_map = {'O': '0', 'Q': '0', 'D': '0', 'I': '1', 'L': '1', 'Z': '2', 'E': '3', 'A': '4', 'S': '5', 'G': '6', 'B': '8', 'T': '7'}
        letter_map = {'0': 'O', '1': 'I', '2': 'Z', '3': 'E', '4': 'A', '5': 'S', '6': 'G', '7': 'T', '8': 'B', '9': 'P'}

        chars_num = [digit_map.get(c, c) for c in part_num]
        chars_let = [letter_map.get(c, c) for c in part_let]

        resultado_final = "".join(chars_num + chars_let)
        return resultado_final if len(resultado_final) == 7 else "OCR_ERROR"

    except Exception:
        return "OCR_ERROR"


object_id_counter = 0
prev_centroids = {}


def get_centroid(box):
    x1, y1, x2, y2 = map(int, box)
    return int((x1 + x2) / 2), int((y1 + y2) / 2)


def assign_id(cx, cy, centroids_dict, threshold=50):
    global object_id_counter
    for oid, (pcx, pcy) in centroids_dict.items():
        if np.sqrt((cx - pcx) ** 2 + (cy - pcy) ** 2) < threshold:
            centroids_dict[oid] = (cx, cy)
            return oid
    object_id_counter += 1
    centroids_dict[object_id_counter] = (cx, cy)
    return object_id_counter


def draw_coco_boxes(results, frame, color=(0, 255, 0), filter_classes=None):
    for r in results:
        if r.boxes is None:
            continue
        for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
            cls = int(cls)
            if filter_classes and cls not in filter_classes:
                continue
            x1, y1, x2, y2 = map(int, box)
            label = f"{r.names[cls]} {float(conf):.2f}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(
                frame,
                label,
                (x1, max(20, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2,
            )
    return frame


cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo cargar el vídeo en: {video_path}")

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_writer = cv2.VideoWriter(
    output_video_path, fourcc, fps, (frame_width, frame_height)
)

csv_headers = [
    "fotograma",
    "tipo_objeto",
    "confianza",
    "id_tracking",
    "x1",
    "y1",
    "x2",
    "y2",
    "matricula_en_su_caso",
    "conf_matricula",
    "mx1",
    "my1",
    "mx2",
    "my2",
    "texto_OCR",
]

f = open(csv_file_path, mode="w", newline="", encoding="utf-8")
writer = csv.writer(f)
writer.writerow(csv_headers)

frame_num = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_num += 1
    if frame_num % 50 == 0:
        print(f"Procesando frame {frame_num}...")

    coco_results = coco_model.predict(
        frame, imgsz=640, conf=0.25, verbose=False, device=0
    )

    for r in coco_results:
        if r.boxes is None:
            continue
        for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
            cls = int(cls)
            if cls not in coco_class_indices:
                continue

            x1, y1, x2, y2 = map(int, box)
            cx, cy = get_centroid((x1, y1, x2, y2))
            oid = assign_id(cx, cy, prev_centroids)
            tipo_obj = r.names[cls]

            writer.writerow(
                [
                    frame_num,
                    tipo_obj,
                    f"{float(conf):.3f}",
                    oid,
                    x1,
                    y1,
                    x2,
                    y2,
                    "",
                    "",
                    "",
                    "",
                    "",
                    "",
                    "",
                ]
            )

    custom_results = custom_model.predict(
        frame, imgsz=640, conf=0.25, verbose=False, device=0
    )

    for r in custom_results:
        if r.boxes is None:
            continue
        for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
            x1, y1, x2, y2 = map(int, box)

            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(frame_width, x2), min(frame_height, y2)

            cx, cy = get_centroid((x1, y1, x2, y2))
            oid = assign_id(cx, cy, prev_centroids)

            plate_crop = frame[y1:y2, x1:x2]
            texto_ocr = recognize_plate_ocr(plate_crop, ocr_reader)

            writer.writerow(
                [
                    frame_num,
                    r.names[int(cls)],
                    f"{float(conf):.3f}",
                    oid,
                    x1,
                    y1,
                    x2,
                    y2,
                    "plate",
                    f"{float(conf):.3f}",
                    x1,
                    y1,
                    x2,
                    y2,
                    texto_ocr,
                ]
            )

            label = f"{r.names[int(cls)]} {float(conf):.2f} | OCR: {texto_ocr}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 255), 2)
            cv2.putText(
                frame,
                label,
                (x1, max(20, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (255, 0, 255),
                2,
            )

    frame = draw_coco_boxes(
        coco_results, frame, color=(0, 255, 0), filter_classes=coco_class_indices
    )

    out_writer.write(frame)

cap.release()
out_writer.release()
f.close()
print("Procesamiento completado con éxito.")

In [ ]:
try:
    import ultralytics.nn.tasks
    torch.serialization.add_safe_globals([ultralytics.nn.tasks.DetectionModel])
except Exception as e:
    print(f"Nota: No se pudo registrar la clase global: {e}")

base_dir = r"" # Al que lo ejecute, poner el directorio raíz donde guarde los archivos
video_path = os.path.join(base_dir, "C0142.MP4")

yolo_model_path = os.path.join(
    base_dir,
    "bestYOLO.pt",
)

output_folder = os.path.join("out")
os.makedirs(output_folder, exist_ok=True)
output_video_path = os.path.join(
    output_folder, "video_con_detecciones_smolvlm_yolo.mp4"
)
csv_file_path = os.path.join(output_folder, "resultsSmolVLM_yolo.csv")

# Cargamos los modelos YOLO
custom_model = YOLO(yolo_model_path)

print("Cargando detector COCO (YOLO11-Large)...")
coco_model = YOLO("yolo11l.pt")

# Configuración y carga de SmolVLM para el análisis de texto de matrículas
print("Cargando modelo SmolVLM...")
smolvlm_model_id = "HuggingFaceTB/SmolVLM-256M-Instruct"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if "cuda" in device else torch.float32

processor = AutoProcessor.from_pretrained(smolvlm_model_id)
smolvlm_model = AutoModelForImageTextToText.from_pretrained(
    smolvlm_model_id, torch_dtype=torch_dtype, attn_implementation="eager"
).to(device).eval()

PROMPT = (
    "Read the vehicle license plate. Return only the letters and digits, without "
    "explanation. For two rows, read the top row first, then the bottom row. "
    "If unreadable, return an empty string."
)

print("Modelos inicializados correctamente.")

person_class_index = 0
vehicle_class_indices = [2, 3, 5, 7]
temp_coco_class_indices = [person_class_index] + vehicle_class_indices

def limitar_caja(caja, ancho, alto):
    x1, y1, x2, y2 = map(int, caja)
    return (
        max(0, min(ancho, x1)),
        max(0, min(alto, y1)),
        max(0, min(ancho, x2)),
        max(0, min(alto, y2)),
    )

def normalizar_texto(texto):
    return re.sub(r'[^A-Z0-9]', '', texto.upper())

def recognize_plate_smolvlm(plate_image):
    try:
        if plate_image is None or plate_image.size == 0:
            return "OCR_ERROR"

        # Convertir recorte OpenCV (BGR) a imagen PIL (RGB)
        pil_img = Image.fromarray(cv2.cvtColor(plate_image, cv2.COLOR_BGR2RGB))

        messages = [{
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": PROMPT}
            ]
        }]

        prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(text=prompt, images=[pil_img], return_tensors="pt")
        inputs = inputs.to(device, dtype=torch_dtype)

        with torch.inference_mode():
            output = smolvlm_model.generate(
                **inputs,
                max_new_tokens=32,
                do_sample=False,
                use_cache=True
            )

        new_tokens = output[:, inputs["input_ids"].shape[1]:]
        raw_text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()

        clean = normalizar_texto(raw_text)

        if len(clean) < 7:
            return "OCR_ERROR"

        return clean if len(clean) == 7 else clean[-7:]

    except Exception:
        return "OCR_ERROR"

object_id_counter = 0
prev_centroids = {}

def get_centroid(box):
    x1, y1, x2, y2 = map(int, box)
    return int((x1 + x2) / 2), int((y1 + y2) / 2)

def assign_id(cx, cy, centroids_dict, threshold=50):
    global object_id_counter
    for oid, (pcx, pcy) in centroids_dict.items():
        if np.sqrt((cx - pcx) ** 2 + (cy - pcy) ** 2) < threshold:
            centroids_dict[oid] = (cx, cy)
            return oid
    object_id_counter += 1
    centroids_dict[object_id_counter] = (cx, cy)
    return object_id_counter

def draw_coco_boxes(results, frame, color=(0, 255, 0), filter_classes=None):
    for r in results:
        if r.boxes is None:
            continue
        for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
            cls = int(cls)
            if filter_classes and cls not in filter_classes:
                continue
            x1, y1, x2, y2 = map(int, box)
            label = f"{r.names[cls]} {float(conf):.2f}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(
                frame,
                label,
                (x1, max(20, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2,
            )
    return frame

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo cargar el vídeo en: {video_path}")

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_writer = cv2.VideoWriter(
    output_video_path, fourcc, fps, (frame_width, frame_height)
)

csv_headers = [
    "fotograma",
    "tipo_objeto",
    "confianza",
    "id_tracking",
    "x1",
    "y1",
    "x2",
    "y2",
    "matricula_en_su_caso",
    "conf_matricula",
    "mx1",
    "my1",
    "mx2",
    "my2",
    "texto_OCR",
]

f = open(csv_file_path, mode="w", newline="", encoding="utf-8")
writer = csv.writer(f)
writer.writerow(csv_headers)

frame_num = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_num += 1
    if frame_num % 50 == 0:
        print(f"Procesando frame {frame_num}...")

    coco_results = coco_model.predict(
        frame, imgsz=640, conf=0.25, verbose=False, device=0
    )

    for r in coco_results:
        if r.boxes is None:
            continue
        for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
            cls = int(cls)
            if cls not in temp_coco_class_indices:
                continue

            x1, y1, x2, y2 = map(int, box)
            cx, cy = get_centroid((x1, y1, x2, y2))
            oid = assign_id(cx, cy, prev_centroids)
            tipo_obj = r.names[cls]

            writer.writerow(
                [
                    frame_num,
                    tipo_obj,
                    f"{float(conf):.3f}",
                    oid,
                    x1,
                    y1,
                    x2,
                    y2,
                    "",
                    "",
                    "",
                    "",
                    "",
                    "",
                    "",
                ]
            )

    custom_results = custom_model.predict(
        frame, imgsz=640, conf=0.25, verbose=False, device=0
    )

    for r in custom_results:
        if r.boxes is None:
            continue
        for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
            x1, y1, x2, y2 = map(int, box)

            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(frame_width, x2), min(frame_height, y2)

            cx, cy = get_centroid((x1, y1, x2, y2))
            oid = assign_id(cx, cy, prev_centroids)

            plate_crop = frame[y1:y2, x1:x2]
            texto_ocr = recognize_plate_smolvlm(plate_crop)

            writer.writerow(
                [
                    frame_num,
                    r.names[int(cls)],
                    f"{float(conf):.3f}",
                    oid,
                    x1,
                    y1,
                    x2,
                    y2,
                    "plate",
                    f"{float(conf):.3f}",
                    x1,
                    y1,
                    x2,
                    y2,
                    texto_ocr,
                ]
            )

            label = f"{r.names[int(cls)]} {float(conf):.2f} | VLM: {texto_ocr}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
            cv2.putText(
                frame,
                label,
                (x1, max(20, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 0, 255),
                2,
            )

    frame = draw_coco_boxes(
        coco_results, frame, color=(0, 255, 0), filter_classes=temp_coco_class_indices
    )

    out_writer.write(frame)

cap.release()
out_writer.release()
f.close()
print("Procesamiento completado con éxito.")

In [ ]:
try:
    import ultralytics.nn.tasks
    torch.serialization.add_safe_globals([ultralytics.nn.tasks.RTDETRDetectionModel])
except Exception as e:
    print(f"Nota: No se pudo registrar la clase global: {e}")

base_dir = r"" # Al que lo ejecute, poner el directorio raíz donde guarde los archivos
video_path = os.path.join(base_dir, "C0142.MP4")

rtdetr_model_path = os.path.join(
    base_dir,
    "bestRT.pt",
)

output_folder = os.path.join("out")
os.makedirs(output_folder, exist_ok=True)
output_video_path = os.path.join(
    output_folder, "video_con_detecciones_smolvlm_rtdetr.mp4"
)
csv_file_path = os.path.join(output_folder, "resultsSmolVLM_rtdetr.csv")

# Cargamos el modelo RT-DETR para las matrículas/personalizados
custom_model = RTDETR(rtdetr_model_path)

print("Cargando detector COCO (YOLO11-Large)...")
coco_model = YOLO("yolo11l.pt")

# Configuración y carga de SmolVLM para el análisis de texto de matrículas
print("Cargando modelo SmolVLM...")
smolvlm_model_id = "HuggingFaceTB/SmolVLM-256M-Instruct"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if "cuda" in device else torch.float32

processor = AutoProcessor.from_pretrained(smolvlm_model_id)
smolvlm_model = AutoModelForImageTextToText.from_pretrained(
    smolvlm_model_id, torch_dtype=torch_dtype, attn_implementation="eager"
).to(device).eval()

PROMPT = (
    "Read the vehicle license plate. Return only the letters and digits, without "
    "explanation. For two rows, read the top row first, then the bottom row. "
    "If unreadable, return an empty string."
)

print("Modelos inicializados correctamente.")

person_class_index = 0
vehicle_class_indices = [2, 3, 5, 7]
coco_class_indices = [person_class_index] + vehicle_class_indices

def limitar_caja(caja, ancho, alto):
    x1, y1, x2, y2 = map(int, caja)
    return (
        max(0, min(ancho, x1)),
        max(0, min(alto, y1)),
        max(0, min(ancho, x2)),
        max(0, min(alto, y2)),
    )

def normalizar_texto(texto):
    return re.sub(r'[^A-Z0-9]', '', texto.upper())

def recognize_plate_smolvlm(plate_image):
    try:
        if plate_image is None or plate_image.size == 0:
            return "OCR_ERROR"

        # Convertir recorte OpenCV (BGR) a imagen PIL (RGB)
        pil_img = Image.fromarray(cv2.cvtColor(plate_image, cv2.COLOR_BGR2RGB))

        messages = [{
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": PROMPT}
            ]
        }]

        prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(text=prompt, images=[pil_img], return_tensors="pt")
        inputs = inputs.to(device, dtype=torch_dtype)

        with torch.inference_mode():
            output = smolvlm_model.generate(
                **inputs,
                max_new_tokens=32,
                do_sample=False,
                use_cache=True
            )

        new_tokens = output[:, inputs["input_ids"].shape[1]:]
        raw_text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()

        clean = normalizar_texto(raw_text)

        if len(clean) < 7:
            return "OCR_ERROR"

        return clean if len(clean) == 7 else clean[-7:]

    except Exception:
        return "OCR_ERROR"

object_id_counter = 0
prev_centroids = {}

def get_centroid(box):
    x1, y1, x2, y2 = map(int, box)
    return int((x1 + x2) / 2), int((y1 + y2) / 2)

def assign_id(cx, cy, centroids_dict, threshold=50):
    global object_id_counter
    for oid, (pcx, pcy) in centroids_dict.items():
        if np.sqrt((cx - pcx) ** 2 + (cy - pcy) ** 2) < threshold:
            centroids_dict[oid] = (cx, cy)
            return oid
    object_id_counter += 1
    centroids_dict[object_id_counter] = (cx, cy)
    return object_id_counter

def draw_coco_boxes(results, frame, color=(0, 255, 0), filter_classes=None):
    for r in results:
        if r.boxes is None:
            continue
        for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
            cls = int(cls)
            if filter_classes and cls not in filter_classes:
                continue
            x1, y1, x2, y2 = map(int, box)
            label = f"{r.names[cls]} {float(conf):.2f}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(
                frame,
                label,
                (x1, max(20, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                2,
            )
    return frame

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo cargar el vídeo en: {video_path}")

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_writer = cv2.VideoWriter(
    output_video_path, fourcc, fps, (frame_width, frame_height)
)

csv_headers = [
    "fotograma",
    "tipo_objeto",
    "confianza",
    "id_tracking",
    "x1",
    "y1",
    "x2",
    "y2",
    "matricula_en_su_caso",
    "conf_matricula",
    "mx1",
    "my1",
    "mx2",
    "my2",
    "texto_OCR",
]

f = open(csv_file_path, mode="w", newline="", encoding="utf-8")
writer = csv.writer(f)
writer.writerow(csv_headers)

frame_num = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_num += 1
    if frame_num % 50 == 0:
        print(f"Procesando frame {frame_num}...")

    coco_results = coco_model.predict(
        frame, imgsz=640, conf=0.25, verbose=False, device=0
    )

    for r in coco_results:
        if r.boxes is None:
            continue
        for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
            cls = int(cls)
            if cls not in coco_class_indices:
                continue

            x1, y1, x2, y2 = map(int, box)
            cx, cy = get_centroid((x1, y1, x2, y2))
            oid = assign_id(cx, cy, prev_centroids)
            tipo_obj = r.names[cls]

            writer.writerow(
                [
                    frame_num,
                    tipo_obj,
                    f"{float(conf):.3f}",
                    oid,
                    x1,
                    y1,
                    x2,
                    y2,
                    "",
                    "",
                    "",
                    "",
                    "",
                    "",
                    "",
                ]
            )

    custom_results = custom_model.predict(
        frame, imgsz=640, conf=0.25, verbose=False, device=0
    )

    for r in custom_results:
        if r.boxes is None:
            continue
        for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
            x1, y1, x2, y2 = map(int, box)

            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(frame_width, x2), min(frame_height, y2)

            cx, cy = get_centroid((x1, y1, x2, y2))
            oid = assign_id(cx, cy, prev_centroids)

            plate_crop = frame[y1:y2, x1:x2]
            texto_ocr = recognize_plate_smolvlm(plate_crop)

            writer.writerow(
                [
                    frame_num,
                    r.names[int(cls)],
                    f"{float(conf):.3f}",
                    oid,
                    x1,
                    y1,
                    x2,
                    y2,
                    "plate",
                    f"{float(conf):.3f}",
                    x1,
                    y1,
                    x2,
                    y2,
                    texto_ocr,
                ]
            )

            label = f"{r.names[int(cls)]} {float(conf):.2f} | VLM: {texto_ocr}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 255), 2)
            cv2.putText(
                frame,
                label,
                (x1, max(20, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (255, 0, 255),
                2,
            )

    frame = draw_coco_boxes(
        coco_results, frame, color=(0, 255, 0), filter_classes=coco_class_indices
    )

    out_writer.write(frame)

cap.release()
out_writer.release()
f.close()
print("Procesamiento completado con éxito.")

#Ejercicio con los Extras

In [ ]:

def anonymize_roi(image, x1, y1, x2, y2, method="blur"):
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(image.shape[1], x2), min(image.shape[0], y2)
        roi = image[y1:y2, x1:x2]
        
        if roi.size == 0:
            return image

        if method == "pixelate":
            h, w = roi.shape[:2]
            small = cv2.resize(roi, (max(1, w // 10), max(1, h // 10)), interpolation=cv2.INTER_LINEAR)
            image[y1:y2, x1:x2] = cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)
        else:
            kw = max(15, (x2 - x1) // 3 | 1)
            kh = max(15, (y2 - y1) // 3 | 1)
            image[y1:y2, x1:x2] = cv2.GaussianBlur(roi, (kw, kh), 0)
            
        return image

def detect_plate_contours(vehicle_crop):
    """
    - Rama A (Morfología / ALPR): Black-Hat + Sobel X + Cierre morfológico + Otsu
    - Rama B (Contornos / Canny): Filtro bilateral + Auto-Canny + closing
    """
    vh, vw = vehicle_crop.shape[:2]
    fallback_box = (
        max(0, int(vw * 0.30)), max(0, int(vh * 0.68)),
        min(vw, max(int(vw * 0.30) + 1, int(vw * 0.70))),
        min(vh, max(int(vh * 0.68) + 1, int(vh * 0.85))),
    )

    if vw < 40 or vh < 40:
        return 1, 0.10, fallback_box

    y_offset = int(vh * 0.40)
    lower_crop = vehicle_crop[y_offset:, :]
    lh, lw = lower_crop.shape[:2]

    if lw < 20 or lh < 15:
        return 1, 0.10, fallback_box

    target_w = 420.0
    scale = target_w / lw
    resized = cv2.resize(
        lower_crop,
        (int(target_w), max(1, int(lh * scale))),
        interpolation=cv2.INTER_AREA,
    )
    rh, rw = resized.shape[:2]

    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(resized, cv2.COLOR_BGR2HSV)

    brillo_medio = cv2.mean(hsv[:, :, 2])[0]
    umbral_v = int(max(95, min(175, brillo_medio * 1.10)))
    mascara_clara = cv2.inRange(hsv, (0, 0, umbral_v), (180, 110, 255))

    # RAMA A
    rect_kern = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 5))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, rect_kern)

    grad_x = cv2.Sobel(blackhat, ddepth=cv2.CV_32F, dx=1, dy=0, ksize=-1)
    grad_x = np.absolute(grad_x)
    max_val = np.max(grad_x)
    if max_val > 0:
        grad_x = (255 * (grad_x / max_val)).astype("uint8")
    else:
        grad_x = grad_x.astype("uint8")

    grad_blur = cv2.GaussianBlur(grad_x, (5, 5), 0)
    grad_closed = cv2.morphologyEx(grad_blur, cv2.MORPH_CLOSE, rect_kern)
    _, thresh_otsu = cv2.threshold(grad_closed, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)

    thresh_otsu = cv2.erode(thresh_otsu, None, iterations=1)
    thresh_otsu = cv2.dilate(thresh_otsu, None, iterations=2)
    clara_dilatada = cv2.dilate(mascara_clara, None, iterations=2)
    thresh_otsu = cv2.bitwise_and(thresh_otsu, thresh_otsu, mask=clara_dilatada)
    thresh_otsu = cv2.dilate(thresh_otsu, None, iterations=1)

    contornos_morph, _ = cv2.findContours(thresh_otsu, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contornos_morph = sorted(contornos_morph, key=cv2.contourArea, reverse=True)[:15]

    # RAMA B
    filtrada = cv2.bilateralFilter(gray, d=9, sigmaColor=25, sigmaSpace=25)
    mediana = np.median(filtrada)
    canny_low = int(max(15, 0.67 * mediana))
    canny_high = int(min(255, 1.33 * mediana))
    bordes = cv2.Canny(filtrada, canny_low, canny_high)

    bordes_cerrados = cv2.morphologyEx(bordes, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
    contornos_canny, _ = cv2.findContours(bordes_cerrados, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    contornos_canny = sorted(contornos_canny, key=cv2.contourArea, reverse=True)[:25]

    candidatos = []

    def evaluar_contorno(c, origen):
        x, y, w, h = cv2.boundingRect(c)
        if h < 8 or w < 24:
            return None

        aspect_ratio = w / float(h)
        if not (1.7 <= aspect_ratio <= 6.2 and (0.11 * rw <= w <= 0.72 * rw)):
            return None

        area_contorno = cv2.contourArea(c)
        area_rect = float(w * h)
        rectangularidad = area_contorno / area_rect if area_rect > 0 else 0.0
        if rectangularidad < 0.42:
            return None

        roi_clara = mascara_clara[y : y + h, x : x + w]
        roi_grad = grad_x[y : y + h, x : x + w]
        roi_bordes = bordes[y : y + h, x : x + w]
        area_roi = float(roi_clara.size)
        if area_roi <= 0:
            return None

        ratio_claro = cv2.countNonZero(roi_clara) / area_roi
        if not (0.35 <= ratio_claro <= 0.96):
            return None

        densidad_grad = np.mean(roi_grad) / 255.0
        densidad_bordes = cv2.countNonZero(roi_bordes) / area_roi
        if densidad_grad < 0.08 and densidad_bordes < 0.04:
            return None

        center_x = x + (w / 2.0)
        dist_centro = abs(center_x - (rw / 2.0)) / (rw / 2.0)
        factor_centralidad = max(0.45, 1.0 - 0.45 * dist_centro)

        bonus_ar = 1.08 if (3.2 <= aspect_ratio <= 5.4 or 1.8 <= aspect_ratio <= 2.5) else 1.0

        perimetro = cv2.arcLength(c, True)
        aprox = cv2.approxPolyDP(c, 0.025 * perimetro, True)
        bonus_poly = 1.08 if (origen == "canny" and len(aprox) == 4) else 1.0

        score = ((0.35 * rectangularidad + 0.35 * ratio_claro + 0.30 * min(1.0, densidad_grad * 2.5))
                 * factor_centralidad * bonus_ar * bonus_poly)

        pad_x = int(w * 0.04)
        pad_y = int(h * 0.06)
        rx1 = max(0, x - pad_x)
        ry1 = max(0, y - pad_y)
        rx2 = min(rw, x + w + pad_x)
        ry2 = min(rh, y + h + pad_y)
        return (rx1, ry1, rx2, ry2, score, origen)

    for c in contornos_morph:
        res = evaluar_contorno(c, "morph")
        if res is not None:
            candidatos.append(res)

    for c in contornos_canny:
        res = evaluar_contorno(c, "canny")
        if res is not None:
            candidatos.append(res)

    if not candidatos:
        return 1, 0.10, fallback_box

    def calcular_iou(boxA, boxB):
        ax1, ay1, ax2, ay2 = boxA[:4]
        bx1, by1, bx2, by2 = boxB[:4]
        ix1, iy1 = max(ax1, bx1), max(ay1, by1)
        ix2, iy2 = min(ax2, bx2), min(ay2, by2)
        inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
        if inter == 0: return 0.0
        areaA = (ax2 - ax1) * (ay2 - ay1)
        areaB = (bx2 - bx1) * (by2 - by1)
        return inter / float(areaA + areaB - inter)

    mejor_caja = None
    mejor_score = 0.0

    for i, cand_a in enumerate(candidatos):
        score_final = cand_a[4]
        for j, cand_b in enumerate(candidatos):
            if i != j and cand_a[5] != cand_b[5]:
                if calcular_iou(cand_a, cand_b) > 0.45:
                    score_final += 0.15
                    break
        if score_final > mejor_score:
            mejor_score = score_final
            mejor_caja = cand_a[:4]

    rx1, ry1, rx2, ry2 = mejor_caja
    px1 = max(0, int(rx1 / scale))
    py1 = max(0, int((ry1 / scale) + y_offset))
    px2 = min(vw, max(px1 + 1, int(rx2 / scale)))
    py2 = min(vh, max(py1 + 1, int((ry2 / scale) + y_offset)))

    conf_final = round(min(0.99, max(0.25, mejor_score)), 3)
    return 1, conf_final, (px1, py1, px2, py2)

track_history = defaultdict(list)
counted_ids = set()
flow_counts = {"Izquierda": 0, "Derecha": 0, "Fondo": 0}

def analyze_flow(track_id, cls_name, cx, cy, width, height):
    track_history[track_id].append((cx, cy))
    
    max_history = 60 if cls_name == "person" else 30
    if len(track_history[track_id]) > max_history:
        track_history[track_id].pop(0)
        
    if track_id in counted_ids:
        return
        
    if len(track_history[track_id]) < 15:
        return
        
    start_x, start_y = track_history[track_id][0]
    dx = cx - start_x
    dy = cy - start_y
    
    thresh_x = 10 if cls_name == "person" else 20
    thresh_y = 10 if cls_name == "person" else 20

    if cy < height * 0.48 and dy < -thresh_y:
        flow_counts["Fondo"] += 1
        counted_ids.add(track_id)
        return

    in_zone_left = (cx < width * 0.45) and (cx < width * 0.15 or cy > height * 0.85)
    
    is_moving_left_or_down = (dx < -thresh_x) or (dy > thresh_y)
    
    if in_zone_left and is_moving_left_or_down:
        flow_counts["Izquierda"] += 1
        counted_ids.add(track_id)
        return

    in_zone_right = (cx > width * 0.55) and (cx > width * 0.85 or cy > height * 0.85)
    
    is_moving_right_or_down = (dx > thresh_x) or (dy > thresh_y)
    
    if in_zone_right and is_moving_right_or_down:
        flow_counts["Derecha"] += 1
        counted_ids.add(track_id)
        return

general_detector = YOLO("yolo11l.pt")

TARGET_CLASSES = ["person", "car", "motorcycle", "bus", "truck"]
VEHICLE_CLASSES = ["car", "motorcycle", "bus", "truck"]

cap = cv2.VideoCapture(r"C:\Users\alejandro\Desktop\ULPGC\4_Carrera\VC\Practicas\P4\P4\C0142.MP4")
if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir el vídeo: {VIDEO_PATH}")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_video = cv2.VideoWriter(r"C:\Users\alejandro\Desktop\ULPGC\4_Carrera\VC\Practicas\P4\P4\salidas_p4\resultado_tracking.mp4", fourcc, fps, (width, height))

unique_objects = {cls_name: set() for cls_name in TARGET_CLASSES}
plate_memory = {}
frame_idx = 0


print(f"Resolución: {width}x{height} | FPS: {fps} | Total fotogramas: {total_frames}")

with tqdm(total=total_frames, unit="frame") as pbar:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1

        track_results = general_detector.track(
            frame, persist=True, verbose=False, device=0
        )[0]

        if track_results.boxes is not None and track_results.boxes.id is not None:
            boxes = track_results.boxes.xyxy.cpu().numpy().astype(int)
            classes = track_results.boxes.cls.cpu().numpy().astype(int)
            confs = track_results.boxes.conf.cpu().numpy()
            track_ids = track_results.boxes.id.cpu().numpy().astype(int)

            for box, cls_idx, conf, track_id in zip(boxes, classes, confs, track_ids):
                cls_name = general_detector.names[cls_idx]
                if cls_name not in TARGET_CLASSES:
                    continue

                unique_objects[cls_name].add(track_id)
                x1, y1, x2, y2 = box
                cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

                analyze_flow(track_id, cls_name, cx, cy, width, height)

                has_plate = 0
                conf_plate = ""
                mx1, my1, mx2, my2 = "", "", "", ""

                if cls_name in VEHICLE_CLASSES:
                    vx1, vy1 = max(0, x1), max(0, y1)
                    vx2, vy2 = min(width, x2), min(height, y2)
                    vehicle_crop = frame[vy1:vy2, vx1:vx2]

                    if vehicle_crop.size > 0:
                        vh, vw = vehicle_crop.shape[:2]
                        has_plate, conf_plate_val, (px1, py1, px2, py2) = detect_plate_contours(vehicle_crop)
                        conf_plate = conf_plate_val

                        if conf_plate >= 0.55:
                            plate_memory[track_id] = (
                                px1 / vw, py1 / vh, px2 / vw, py2 / vh, conf_plate
                            )
                        elif conf_plate < 0.45 and track_id in plate_memory:
                            rx1, ry1, rx2, ry2, prev_conf = plate_memory[track_id]
                            px1 = max(0, int(rx1 * vw))
                            py1 = max(0, int(ry1 * vh))
                            px2 = min(vw, max(px1 + 1, int(rx2 * vw)))
                            py2 = min(vh, max(py1 + 1, int(ry2 * vh)))
                            conf_plate = round(prev_conf * 0.90, 3)

                        mx1, my1 = vx1 + px1, vy1 + py1
                        mx2, my2 = vx1 + px2, vy1 + py2

                color = (0, 255, 0) if cls_name == "person" else (255, 120, 0)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(
                    frame, f"ID:{track_id} {cls_name}", (x1, max(15, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2
                )

                if cls_name == "person":
                    frame = anonymize_roi(frame, x1, y1, x2, y2, method="pixelate") # Asumimos definida
                elif has_plate and mx1 != "":
                    cv2.rectangle(frame, (mx1, my1), (mx2, my2), (0, 0, 255), 2)
                    frame = anonymize_roi(frame, mx1, my1, mx2, my2, method="blur")

        y_pos = 30
        cv2.putText(
            frame, "TOTALES ACUMULADOS:", (15, y_pos),
            cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 255), 2
        )
        for cname, ids in unique_objects.items():
            y_pos += 22
            cv2.putText(
                frame, f" - {cname}: {len(ids)}", (20, y_pos),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2
            )

        y_pos += 30
        cv2.putText(
            frame, "FLUJO DE SALIDA:", (15, y_pos),
            cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2
        )
        for direction, count in flow_counts.items():
            y_pos += 22
            cv2.putText(
                frame, f" - {direction}: {count}", (20, y_pos),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2
            )

        out_video.write(frame)
        pbar.update(1)

cap.release()
out_video.release()

Resolución: 1920x1080 | FPS: 25.0 | Total fotogramas: 2832


100%|██████████| 2832/2832 [06:41<00:00,  7.06frame/s]



Procesamiento finalizado con éxito.
